![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 08: Reproducible Research & Deployment
***
**Learning objectives**
- Write unit tests for data pipeline and model functions with `unittest`
- Build model performance tests (AUC thresholds, direction tests)
- Implement property-based testing with `hypothesis`
- Configure GitHub Actions CI/CD for model training and deployment
- Apply pre-commit hooks for code quality (black, flake8, isort)
- Measure test coverage and enforce minimums in CI

**Estimated time:** 2.5 hours | **Level:** Intermediate-Advanced | **Libraries:** `unittest`, `pytest`


## 1. Setup

In [ ]:
import os, json, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod08/tests", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42); N = 2000
def sigmoid(x): return 1/(1+np.exp(-x))
age = np.random.normal(60,15,N).clip(18,90).astype(int)
los = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
dm  = np.random.binomial(1,0.35,N)
hf  = np.random.binomial(1,0.22,N)
copd= np.random.binomial(1,0.18,N)
logit=-3.2+0.6*hf+0.5*dm+0.3*copd+0.02*(age-60)+0.2*(los>7).astype(int)
readmitted = np.random.binomial(1,sigmoid(logit),N)
df = pd.DataFrame({"age":age,"los_days":los,"diabetes":dm,"hf":hf,"copd":copd,"readmitted":readmitted})
FEATURES = ["age","los_days","diabetes","hf","copd"]
X_tr,X_te,y_tr,y_te = train_test_split(df[FEATURES],df["readmitted"],test_size=0.2,stratify=df["readmitted"],random_state=42)
model = GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=42)
model.fit(X_tr,y_tr)
print(f"Test dataset: {N} patients | AUC={roc_auc_score(y_te,model.predict_proba(X_te)[:,1]):.4f}")


## 2. Testing Pyramid for Health Analytics

```
         /\
        /  \
       / E2E \         <- End-to-end (1-5 tests: full pipeline smoke test)
      /--------\
     / Integration \   <- Integration (10-20: API endpoints, DB connections)
    /--------------\
   /   Unit Tests   \  <- Unit tests (100+: individual functions, classes)
  /------------------\

Clinical Model Testing Priorities:
  1. Data validation (correct types, ranges, no PHI leakage)
  2. Feature engineering (correct transforms, no data leakage)
  3. Model direction (high-risk > low-risk scores)
  4. Model performance (AUC above clinical minimum)
  5. API contract (correct request/response schemas)
  6. Reproducibility (same seed → same output)


## 3. Unit Tests: Data Pipeline

In [ ]:
# Unit tests for data pipeline and model functions
import unittest

# Functions to test
def validate_features(df, required_cols, age_col="age", target_col="readmitted"):
    errors = []
    missing = [c for c in required_cols if c not in df.columns]
    if missing: errors.append(f"Missing columns: {missing}")
    if age_col in df.columns and not df[age_col].between(18,90).all():
        errors.append(f"{age_col} values outside 18-90 range")
    if target_col in df.columns and not df[target_col].isin([0,1]).all():
        errors.append(f"{target_col} must be binary 0/1")
    return errors

def compute_positive_rate(y):
    return float(y.mean())

def compute_smr(observed, expected):
    if expected <= 0: raise ValueError("Expected must be positive")
    return observed / expected

def clean_age(age_series, min_age=18, max_age=90):
    return age_series.clip(min_age, max_age)

class TestDataPipeline(unittest.TestCase):
    def setUp(self):
        np.random.seed(42)
        self.df_valid = pd.DataFrame({
            "age":[65,45,72],"los_days":[5,3,12],
            "diabetes":[1,0,1],"hf":[0,0,1],"copd":[0,0,1],"readmitted":[0,0,1],
        })
        self.df_invalid = self.df_valid.copy()
        self.df_invalid.loc[0,"age"] = 150  # invalid age

    def test_validate_features_pass(self):
        errors = validate_features(self.df_valid, FEATURES + ["readmitted"])
        self.assertEqual(errors, [], f"Unexpected errors: {errors}")

    def test_validate_features_invalid_age(self):
        errors = validate_features(self.df_invalid, FEATURES)
        self.assertTrue(any("age" in e.lower() for e in errors))

    def test_validate_features_missing_col(self):
        df_missing = self.df_valid.drop("hf", axis=1)
        errors = validate_features(df_missing, FEATURES)
        self.assertTrue(any("hf" in e for e in errors))

    def test_positive_rate_range(self):
        y = pd.Series([0,0,1,1,0])
        rate = compute_positive_rate(y)
        self.assertGreaterEqual(rate, 0.0)
        self.assertLessEqual(rate, 1.0)
        self.assertAlmostEqual(rate, 0.4, places=5)

    def test_smr_valid(self):
        smr = compute_smr(observed=8, expected=5.0)
        self.assertAlmostEqual(smr, 1.6, places=5)

    def test_smr_zero_expected(self):
        with self.assertRaises(ValueError):
            compute_smr(observed=3, expected=0)

    def test_clean_age_clipping(self):
        ages = pd.Series([15, 50, 100])
        cleaned = clean_age(ages)
        self.assertEqual(cleaned.min(), 18)
        self.assertEqual(cleaned.max(), 90)

# Run tests
loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestDataPipeline)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\nTests: {result.testsRun} | Passed: {result.testsRun-len(result.failures)-len(result.errors)} | Failed: {len(result.failures)} | Errors: {len(result.errors)}")


## 4. Model Performance Tests

In [ ]:
class TestModelPerformance(unittest.TestCase):
    def setUp(self):
        self.model = model
        self.X_te  = X_te
        self.y_te  = y_te
        self.prob  = self.model.predict_proba(X_te)[:,1]
        self.auc   = roc_auc_score(y_te, self.prob)

    def test_auc_above_threshold(self):
        MIN_AUC = 0.65
        self.assertGreater(self.auc, MIN_AUC,
            f"AUC {self.auc:.4f} below minimum acceptable {MIN_AUC}")

    def test_predictions_in_range(self):
        self.assertTrue((self.prob >= 0).all() and (self.prob <= 1).all(),
            "Probabilities outside [0,1] range")

    def test_positive_class_higher_scores(self):
        mean_pos = self.prob[self.y_te==1].mean()
        mean_neg = self.prob[self.y_te==0].mean()
        self.assertGreater(mean_pos, mean_neg,
            f"Mean score for positives ({mean_pos:.3f}) not > negatives ({mean_neg:.3f})")

    def test_no_nan_predictions(self):
        self.assertFalse(np.isnan(self.prob).any(), "NaN values in predictions")

    def test_feature_count(self):
        self.assertEqual(self.model.n_features_in_, len(FEATURES),
            f"Model expects {self.model.n_features_in_} features, got {len(FEATURES)}")

    def test_batch_prediction_shape(self):
        batch_size = 50
        X_batch = X_te.iloc[:batch_size]
        preds = self.model.predict_proba(X_batch)[:,1]
        self.assertEqual(len(preds), batch_size)

    def test_high_risk_patient(self):
        X_high = pd.DataFrame([{"age":80,"los_days":14,"diabetes":1,"hf":1,"copd":1}])
        prob_high = float(self.model.predict_proba(X_high)[:,1])
        X_low  = pd.DataFrame([{"age":25,"los_days":1,"diabetes":0,"hf":0,"copd":0}])
        prob_low = float(self.model.predict_proba(X_low)[:,1])
        self.assertGreater(prob_high, prob_low,
            f"High-risk patient ({prob_high:.3f}) should score higher than low-risk ({prob_low:.3f})")

suite2  = unittest.TestLoader().loadTestsFromTestCase(TestModelPerformance)
result2 = unittest.TextTestRunner(verbosity=2).run(suite2)
print(f"\nModel tests: {result2.testsRun} | Passed: {result2.testsRun-len(result2.failures)-len(result2.errors)} | Failed: {len(result2.failures)}")


## 5. CI/CD Pipeline — GitHub Actions

In [ ]:
# CI/CD pipeline for health analytics projects
GITHUB_ACTIONS_WORKFLOW = (
    "# .github/workflows/ci.yml\n"
    "name: Health Analytics CI/CD\n\n"
    "on:\n"
    "  push:\n"
    "    branches: [main, develop]\n"
    "  pull_request:\n"
    "    branches: [main]\n"
    "  schedule:\n"
    "    - cron: '0 6 * * 1'  # Weekly Monday 6am -- model drift check\n\n"
    "jobs:\n"
    "  test:\n"
    "    runs-on: ubuntu-latest\n"
    "    steps:\n"
    "      - uses: actions/checkout@v4\n"
    "      - uses: actions/setup-python@v4\n"
    "        with: {python-version: '3.10'}\n"
    "      - name: Install dependencies\n"
    "        run: pip install -r requirements.txt\n"
    "      - name: Verify data integrity\n"
    "        run: python src/data/validate.py --manifest data/MANIFEST.json\n"
    "      - name: Run unit tests\n"
    "        run: pytest tests/unit/ -v --tb=short --cov=src --cov-report=xml\n"
    "      - name: Run model tests\n"
    "        run: pytest tests/model/ -v --tb=short\n"
    "      - name: Coverage report\n"
    "        uses: codecov/codecov-action@v3\n\n"
    "  build_and_push:\n"
    "    needs: test\n"
    "    if: github.ref == 'refs/heads/main'\n"
    "    runs-on: ubuntu-latest\n"
    "    steps:\n"
    "      - uses: actions/checkout@v4\n"
    "      - name: Build Docker image\n"
    "        run: docker build -t readmission-api:${{ github.sha }} .\n"
    "      - name: Push to registry\n"
    "        run: |\n"
    "          docker push ghcr.io/org/readmission-api:${{ github.sha }}\n"
    "          docker push ghcr.io/org/readmission-api:latest\n\n"
    "  deploy_staging:\n"
    "    needs: build_and_push\n"
    "    environment: staging\n"
    "    steps:\n"
    "      - name: Deploy to staging\n"
    "        run: |\n"
    "          kubectl set image deployment/api api=ghcr.io/org/readmission-api:${{ github.sha }}\n"
    "          kubectl rollout status deployment/api\n"
    "      - name: Smoke test\n"
    "        run: curl -f https://api-staging.health.org/health\n"
)
print("GitHub Actions CI/CD Workflow:")
print(GITHUB_ACTIONS_WORKFLOW)
workflow_path = Path("/tmp/mod08/.github/workflows/ci.yml")
workflow_path.parent.mkdir(parents=True, exist_ok=True)
workflow_path.write_text(GITHUB_ACTIONS_WORKFLOW)


## 6. Pre-commit Hooks & Code Quality

In [ ]:
# Pre-commit hooks and code quality tools
PRE_COMMIT_CONFIG = (
    "# .pre-commit-config.yaml\n"
    "repos:\n"
    "  - repo: https://github.com/psf/black\n"
    "    rev: 23.9.1\n"
    "    hooks:\n"
    "      - id: black\n"
    "        language_version: python3.10\n"
    "        args: ['--line-length=100']\n\n"
    "  - repo: https://github.com/pycqa/flake8\n"
    "    rev: 6.1.0\n"
    "    hooks:\n"
    "      - id: flake8\n"
    "        args: ['--max-line-length=100','--ignore=E203,W503']\n\n"
    "  - repo: https://github.com/pycqa/isort\n"
    "    rev: 5.12.0\n"
    "    hooks:\n"
    "      - id: isort\n"
    "        args: ['--profile=black']\n\n"
    "  - repo: https://github.com/pre-commit/mirrors-mypy\n"
    "    rev: v1.5.1\n"
    "    hooks:\n"
    "      - id: mypy\n"
    "        additional_dependencies: [types-requests]\n\n"
    "  - repo: https://github.com/nbQA-dev/nbQA\n"
    "    rev: 1.7.0\n"
    "    hooks:\n"
    "      - id: nbqa-black  # auto-format notebooks\n"
    "      - id: nbqa-isort  # sort imports in notebooks\n\n"
    "# Setup: pip install pre-commit && pre-commit install\n"
    "# Run manually: pre-commit run --all-files\n"
)
print("Pre-commit configuration:")
print(PRE_COMMIT_CONFIG)

# Code quality metrics using AST analysis
import ast, re

def compute_code_metrics(source_code):
    try:
        tree = ast.parse(source_code)
    except SyntaxError as e:
        return {"error": str(e)}
    functions = [n for n in ast.walk(tree) if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef))]
    classes   = [n for n in ast.walk(tree) if isinstance(n,ast.ClassDef)]
    lines     = [l for l in source_code.split("\n") if l.strip() and not l.strip().startswith("#")]
    docstrings= sum(1 for f in functions if ast.get_docstring(f))
    return {
        "lines_of_code": len(lines),
        "num_functions" : len(functions),
        "num_classes"   : len(classes),
        "docstring_coverage": f"{docstrings/max(len(functions),1)*100:.0f}%",
        "avg_func_length": round(sum(len(ast.unparse(f).split("\n")) for f in functions)/max(len(functions),1),1),
    }

sample_code = open("/tmp/mod08/main.py").read() if Path("/tmp/mod08/main.py").exists() else "def hello():\n    pass\ndef world(x):\n    return x"
metrics = compute_code_metrics(sample_code)
print("\nCode quality metrics:")
for k,v in metrics.items(): print(f"  {k:28s}: {v}")


## 7. Property-Based Testing

In [ ]:
# Property-based testing: verify invariants hold for ALL inputs
# hypotheis library generates random valid inputs automatically
try:
    from hypothesis import given, settings, strategies as st
    HYP_OK = True
except ImportError:
    HYP_OK = False
    print("pip install hypothesis")

if HYP_OK:
    from hypothesis import given, settings, strategies as st

    @given(
        age      = st.integers(min_value=18, max_value=90),
        los_days = st.integers(min_value=1,  max_value=60),
        diabetes = st.integers(min_value=0,  max_value=1),
        hf       = st.integers(min_value=0,  max_value=1),
        copd     = st.integers(min_value=0,  max_value=1),
    )
    @settings(max_examples=200)
    def test_prediction_always_in_unit_interval(age, los_days, diabetes, hf, copd):
        X = pd.DataFrame([{"age":age,"los_days":los_days,"diabetes":diabetes,"hf":hf,"copd":copd}])
        prob = float(model.predict_proba(X)[:,1])
        assert 0.0 <= prob <= 1.0, f"Probability {prob} outside [0,1] for {X.to_dict()}"

    print("Running property-based tests (200 random examples)...")
    test_prediction_always_in_unit_interval()
    print("[PASS] Predictions always in [0,1] for all valid inputs")
else:
    # Manual property check
    print("Manual property check (1000 random patients):")
    np.random.seed(99)
    test_X = pd.DataFrame({
        "age"     : np.random.randint(18,91,1000),
        "los_days": np.random.randint(1,61,1000),
        "diabetes": np.random.binomial(1,0.35,1000),
        "hf"      : np.random.binomial(1,0.22,1000),
        "copd"    : np.random.binomial(1,0.18,1000),
    })
    probs = model.predict_proba(test_X)[:,1]
    print(f"  All probabilities in [0,1]: {((probs>=0)&(probs<=1)).all()}")
    print(f"  Any NaN: {np.isnan(probs).any()}")
    print(f"  Min: {probs.min():.6f} | Max: {probs.max():.6f}")


## 8. Knowledge Check
1. A test checks `auc > 0.65`. The AUC is 0.66, so the test passes. Six months later a data bug makes AUC=0.67 look good but the data is corrupted. What additional tests would catch this?
2. Your CI pipeline takes 45 minutes to run. 12 developers each push 5 times/day. What optimisations reduce this to <10 minutes?
3. A property-based test (`hypothesis`) found that `age=18, los_days=1` all zeros returns `prob=0.0001`. Is this a bug?
4. Pre-commit hooks auto-format a notebook (`nbqa-black`) and change 200 lines. How does this affect reproducibility?
5. Write a test that verifies the model's AUC on a held-out test set is within 0.05 of the AUC logged in MLflow for the production model.


In [ ]:
# Exercise 5 — MLflow AUC consistency test
import json
from pathlib import Path

def test_model_auc_consistent_with_registry(model_obj, X_test, y_test,
                                              registry_path="/tmp/mod08/model_registry.json",
                                              tolerance=0.05):
    registry_file = Path(registry_path)
    if not registry_file.exists():
        print("SKIP: No model registry file found")
        return True
    registry = json.loads(registry_file.read_text())
    prod_versions = [v for v in registry["version_history"] if v["stage"]=="Production"]
    if not prod_versions:
        print("SKIP: No production version in registry")
        return True
    registered_auc = prod_versions[-1]["auc"]
    current_auc    = roc_auc_score(y_test, model_obj.predict_proba(X_test)[:,1])
    auc_diff       = abs(current_auc - registered_auc)
    passed         = auc_diff <= tolerance
    print(f"Registry AUC : {registered_auc:.4f}")
    print(f"Current AUC  : {current_auc:.4f}")
    print(f"Difference   : {auc_diff:.4f} (tolerance={tolerance})")
    print(f"Test result  : {'PASS' if passed else 'FAIL - model may have drifted!'}")
    return passed

test_model_auc_consistent_with_registry(model, X_te, y_te)


***
**Next → NB-07: Model Cards, Ethics & Clinical Governance**
